# Practice 3: Customer Clustering - Bước 2: Kiểm tra dữ liệu (Data Collection & Validation)

---

## 1. Import các thư viện cần thiết

Chúng ta bắt đầu bằng việc import các thư viện cốt lõi để thao tác dữ liệu: `pandas` và `numpy`.

In [1]:
import pandas as pd
import numpy as np
import os

## 2. Tải tập dữ liệu thô (Raw Data)

Đường dẫn đến thư mục dữ liệu gốc là `../data/raw/`.

In [2]:
train_path = "../data/raw/Train.csv"
train_df = pd.read_csv(train_path)

print(f"Kích thước tập Train: {train_df.shape}")

Kích thước tập Train: (8068, 11)


## 3. Xem trước một số dòng dữ liệu

Quan sát 20 dòng đầu tiên của tập Train để hình dung cấu trúc các đặc trưng.

In [3]:
train_df.head(20)

,ID,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation
0,462809,Male,No,22,No,Healthcare,1.0,Low,4.0,Cat_4,D
1,462643,Female,Yes,38,Yes,Engineer,NaN,Average,3.0,Cat_4,A
2,466315,Female,Yes,67,Yes,Engineer,1.0,Low,1.0,Cat_6,B
3,461735,Male,Yes,67,Yes,Lawyer,0.0,High,2.0,Cat_6,B
4,462669,Female,Yes,40,Yes,Entertainment,NaN,High,6.0,Cat_6,A
5,461319,Male,Yes,56,No,Artist,0.0,Average,2.0,Cat_6,C
6,460156,Male,No,32,Yes,Healthcare,1.0,Low,3.0,Cat_6,C
7,464347,Female,No,33,Yes,Healthcare,1.0,Low,3.0,Cat_6,D
8,465015,Female,Yes,61,Yes,Engineer,0.0,Low,3.0,Cat_7,D
9,465176,Female,Yes,55,Yes,Artist,1.0,Average,4.0,Cat_6,C


## 4. Kiểm tra kiểu dữ liệu của các đặc trưng (Data Types)

Xác định các cột nào là kiểu số (numerical) và cột nào là kiểu phân loại (categorical).

In [4]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8068 entries, 0 to 8067
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ID               8068 non-null   int64  
 1   Gender           8068 non-null   object 
 2   Ever_Married     7928 non-null   object 
 3   Age              8068 non-null   int64  
 4   Graduated        7990 non-null   object 
 5   Profession       7944 non-null   object 
 6   Work_Experience  7239 non-null   float64
 7   Spending_Score   8068 non-null   object 
 8   Family_Size      7733 non-null   float64
 9   Var_1            7992 non-null   object 
 10  Segmentation     8068 non-null   object 
dtypes: float64(2), int64(2), object(7)
memory usage: 693.5+ KB


## 5. Kiểm tra giá trị khuyết thiếu (Missing Values)

Thống kê số lượng và tỷ lệ % khuyết thiếu trên mỗi cột để lập phương án điền khuyết ở bước sau.

In [5]:
def check_missing_values(df):
    missing_count = df.isnull().sum()
    missing_percent = (df.isnull().sum() / len(df)) * 100
    missing_table = pd.concat([missing_count, missing_percent], axis=1, keys=['Missing Count', 'Percentage (%)'])
    return missing_table[missing_table['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False)

print("--- Các cột bị khuyết thiếu ở tập Train ---")
print(check_missing_values(train_df))

--- Các cột bị khuyết thiếu ở tập Train ---
                 Missing Count  Percentage (%)
Work_Experience            829       10.275161
Family_Size                335        4.152206
Ever_Married               140        1.735250
Profession                 124        1.536936
Graduated                   78        0.966782
Var_1                       76        0.941993


## 6. Kiểm tra dòng trùng lặp (Duplicate Rows)

Trùng lặp dữ liệu có thể làm sai lệch phân phối và kết quả phân cụm. Ta kiểm tra xem có dòng nào trùng hoàn toàn (hoặc trùng mã định danh `ID`) hay không.

In [6]:
duplicates_total = train_df.duplicated().sum()
duplicates_id = train_df.duplicated(subset=['ID']).sum()
print(f"Số dòng trùng lặp hoàn toàn: {duplicates_total}")
print(f"Số dòng bị trùng lặp ID: {duplicates_id}")

Số dòng trùng lặp hoàn toàn: 0
Số dòng bị trùng lặp ID: 0


## 7. Khảo sát sơ bộ các giá trị phân loại

Để đảm bảo dữ liệu phân loại sạch, ta kiểm tra các giá trị độc bản (unique values) của từng cột dạng chuỗi.

In [7]:
categorical_cols = train_df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    print(f"Cột {col}: {train_df[col].dropna().unique()}")

Cột Gender: ['Male' 'Female']
Cột Ever_Married: ['No' 'Yes']
Cột Graduated: ['No' 'Yes']
Cột Profession: ['Healthcare' 'Engineer' 'Lawyer' 'Entertainment' 'Artist' 'Executive'
 'Doctor' 'Homemaker' 'Marketing']
Cột Spending_Score: ['Low' 'Average' 'High']
Cột Var_1: ['Cat_4' 'Cat_6' 'Cat_7' 'Cat_3' 'Cat_1' 'Cat_2' 'Cat_5']
Cột Segmentation: ['D' 'A' 'B' 'C']


## 9. Kiểm tra phân phối của nhãn mục tiêu (Segmentation Distribution)

Mặc dù đây là bài toán phân cụm không giám sát và nhãn `Segmentation` không được dùng để huấn luyện, nhưng nhãn này là Ground Truth rất quan trọng giúp ta đánh giá ngoài (External Evaluation) ở bước cuối. Ta khảo sát tỷ lệ phân phối của các phân khúc A, B, C, D để xem dữ liệu có bị mất cân bằng hay không.

In [8]:
# Thống kê số lượng và tỷ lệ % của từng phân khúc
segment_counts = train_df['Segmentation'].value_counts()
segment_percent = train_df['Segmentation'].value_counts(normalize=True) * 100

segment_dist = pd.concat([segment_counts, segment_percent], axis=1, keys=['Số lượng', 'Tỷ lệ (%)'])
print("--- Phân bố nhãn Segmentation trong tập Train ---")
print(segment_dist)

--- Phân bố nhãn Segmentation trong tập Train ---
              Số lượng  Tỷ lệ (%)
Segmentation                     
D                 2268  28.111056
A                 1972  24.442241
C                 1970  24.417452
B                 1858  23.029251


## 8. Mô tả thống kê sơ bộ (Descriptive Statistics)

Thống kê mô tả đối với cả các cột số học.

In [9]:
train_df.describe().T

,count,mean,std,min,25%,50%,75%,max
ID,8068.0,463479.214551,2595.381232,458982.0,461240.75,463472.5,465744.25,467974.0
Age,8068.0,43.466906,16.711696,18.0,30.00,40.0,53.00,89.0
Work_Experience,7239.0,2.641663,3.406763,0.0,0.00,1.0,4.00,14.0
Family_Size,7733.0,2.850123,1.531413,1.0,2.00,3.0,4.00,9.0


## 10. Kiểm tra tính nhất quán logic của dữ liệu (Consistency Checks)

Để đảm bảo chất lượng dữ liệu trước khi huấn luyện mô hình phân cụm, ta tiến hành kiểm tra một số mâu thuẫn logic thực tế có thể tồn tại trong dữ liệu thô.

### 10.1. Mâu thuẫn giữa Độ tuổi và Kinh nghiệm làm việc (Age vs. Work Experience)

Một người đi làm thực tế thường bắt đầu từ 15 tuổi trở lên. Nếu hiệu số `Age - Work_Experience` nhỏ hơn 15, điều đó phản ánh sự bất hợp lý logic (ví dụ: đi làm từ lúc còn quá nhỏ).

In [13]:
# Lọc các trường hợp Age - Work_Experience < 15
work_age_anomaly = train_df[train_df['Age'] - train_df['Work_Experience'] < 15]
print(f"Số lượng bản ghi bất thường (Age - Work_Exp < 15): {len(work_age_anomaly)}")
if len(work_age_anomaly) > 0:
    print("\nVí dụ 5 bản ghi bất thường đầu tiên:")
    print(work_age_anomaly[['ID', 'Age', 'Work_Experience', 'Profession']].head(20))

Số lượng bản ghi bất thường (Age - Work_Exp < 15): 137

Ví dụ 5 bản ghi bất thường đầu tiên:
          ID  Age  Work_Experience  Profession
42    464590   27             14.0      Artist
108   466466   19              6.0  Healthcare
132   464857   18              6.0  Healthcare
176   464866   23             11.0    Engineer
201   466065   19              9.0  Healthcare
367   464527   25             12.0   Executive
368   463560   18              9.0  Healthcare
409   466557   21              7.0  Healthcare
500   465309   18              6.0  Healthcare
531   462314   28             14.0      Doctor
601   466522   18              6.0         NaN
613   465255   21              9.0  Healthcare
702   465368   18              7.0  Healthcare
793   460908   20              7.0  Healthcare
804   467914   18              8.0  Healthcare
843   462047   26             12.0      Artist
905   462040   19              5.0  Healthcare
1092  461035   21              7.0         NaN
1158  464829  

### 10.2. Mâu thuẫn giữa Nghề nghiệp và Trình độ học vấn (Profession vs. Graduation)

Trong thực tế, các ngành nghề như Bác sĩ (Doctor) và Luật sư (Lawyer) bắt buộc nhân sự phải tốt nghiệp Đại học (`Graduated == Yes`). Ta kiểm tra xem có trường hợp nào được ghi nhận chưa tốt nghiệp (`No`) nhưng vẫn làm các nghề này hay không.

In [11]:
# Lọc bác sĩ/luật sư chưa tốt nghiệp
profession_anomaly = train_df[(train_df['Profession'].isin(['Doctor', 'Lawyer'])) & (train_df['Graduated'] == 'No')]
print(f"Số lượng bác sĩ/luật sư chưa tốt nghiệp: {len(profession_anomaly)}")
if len(profession_anomaly) > 0:
    print("\nVí dụ 5 bản ghi đầu tiên:")
    print(profession_anomaly[['ID', 'Profession', 'Graduated']].head(5))

Số lượng bác sĩ/luật sư chưa tốt nghiệp: 518

Ví dụ 5 bản ghi đầu tiên:
        ID Profession Graduated
13  459573     Lawyer        No
14  460849     Doctor        No
31  462216     Doctor        No
34  459861     Lawyer        No
36  465572     Doctor        No


### 10.3. Khảo sát mối tương quan giữa Tình trạng kết hôn và Mức chi tiêu (Ever Married vs. Spending Score)

Ta kiểm tra xem có khách hàng nào chưa từng kết hôn (`Ever_Married == 'No'`) nhưng được đánh giá mức chi tiêu là Trung bình (`Average`) hoặc Cao (`High`) hay không.

In [12]:
# 1. Kiểm tra số lượng người độc thân có chi tiêu Average/High
single_high_spending = train_df[(train_df['Ever_Married'] == 'No') & (train_df['Spending_Score'].isin(['Average', 'High']))]
print(f"Số lượng người độc thân có chi tiêu Average/High: {len(single_high_spending)}")

# 2. Phân tích chéo người độc thân theo Nghề nghiệp và Mức chi tiêu
single_df = train_df[train_df['Ever_Married'] == 'No']
print("\n--- Phân bố chi tiêu của người độc thân theo Nghề nghiệp ---")
print(pd.crosstab(single_df['Profession'], single_df['Spending_Score'], margins=True))

# 3. Phân tích chéo người độc thân theo Quy mô gia đình và Mức chi tiêu
print("\n--- Phân bố chi tiêu của người độc thân theo Quy mô gia đình ---")
print(pd.crosstab(single_df['Family_Size'], single_df['Spending_Score'], margins=True))

Số lượng người độc thân có chi tiêu Average/High: 0

--- Phân bố chi tiêu của người độc thân theo Nghề nghiệp ---
Spending_Score   Low   All
Profession                
Artist           713   713
Doctor           366   366
Engineer         267   267
Entertainment    358   358
Executive         37    37
Healthcare      1153  1153
Homemaker        112   112
Lawyer            40    40
Marketing        189   189
All             3235  3235

--- Phân bố chi tiêu của người độc thân theo Quy mô gia đình ---
Spending_Score   Low   All
Family_Size               
1.0              858   858
2.0              453   453
3.0              630   630
4.0              608   608
5.0              315   315
6.0              120   120
7.0               59    59
8.0               36    36
9.0               22    22
All             3101  3101
